# Prática de Laboratório 11: OpenMP Tasks — Parte 2

**Ambiente:** Google Colab / Jupyter Notebook  
**Linguagem:** C com OpenMP

## Conteúdo

- tarefas em estruturas irregulares, como listas encadeadas;
- dependências `in`, `out` e `inout`;
- diferenças entre `barrier`, `taskwait` e `taskgroup`;
- `taskyield`;
- `taskloop`, `grainsize`, `num_tasks`, `nogroup` e `taskloop simd`.

## Objetivos

Ao final da prática, o estudante deverá ser capaz de criar e sincronizar tarefas, descrever o grafo de dependências, controlar a granularidade e distinguir os principais atributos de compartilhamento de dados.

> As saídas podem variar entre execuções, pois o escalonamento das tarefas é realizado pelo ambiente de execução do OpenMP.


# 1. Lista encadeada e criação dinâmica de tarefas

Uma lista encadeada não possui um número de iterações conhecido previamente. Uma única *thread* pode percorrer a estrutura e criar uma tarefa para cada nó. A cláusula `firstprivate(p)` preserva, em cada tarefa, o ponteiro para o nó encontrado no momento da criação.

In [ ]:
%%writefile lista_tasks.c
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <omp.h>

typedef struct No {  // Estrutura do nó
    int valor;
    struct No *proximo;
} No;

No *inserir_fim(No *inicio, int valor) { // Rotina para inserir um nó no final da lista
    No *novo = malloc(sizeof(No));
    if (!novo) {
        perror("malloc");
        exit(EXIT_FAILURE);
    }
    novo->valor = valor;
    novo->proximo = NULL;

    if (!inicio) return novo;

    No *p = inicio;
    while (p->proximo) p = p->proximo;
    p->proximo = novo;
    return inicio;
}

void trabalho_independente(No *p, int criador) { // Rotina para simular trabalho
    usleep(1000 * (20 + (p->valor % 5) * 20));
    #pragma omp critical
    printf("Nó %2d: criado pela thread %d e executado pela thread %d\n",
           p->valor, criador, omp_get_thread_num());
}

void liberar_lista(No *inicio) { // Rotina para liberar a lista
    while (inicio) {
        No *proximo = inicio->proximo;
        free(inicio);
        inicio = proximo;
    }
}

int main(void) {
    No *inicio = NULL;
    for (int i = 1; i <= 12; i++)
        inicio = inserir_fim(inicio, i);

    No *p = inicio;

    #pragma omp parallel num_threads(4)
    {
        #pragma omp single nowait
        {
            while (p) {
                int criador = omp_get_thread_num();  // Guarda o ID da thread criadora

                #pragma omp task firstprivate(p, criador)
                trabalho_independente(p, criador);   // Cria tarefa independente

                p = p->proximo;  // Avança para o próximo nó
            }
        }
    }

    liberar_lista(inicio);
    return 0;
}


Writing lista_tasks.c


In [ ]:
!gcc -O2 -fopenmp lista_tasks.c -o lista_tasks
!./lista_tasks


### Questões

1. Por que `p` deve ser `firstprivate`?
2. A thread que cria a tarefa é necessariamente a que a executa?
3. Qual barreira garante o término das tarefas quando `single nowait` é utilizado?
4. Remova `nowait`. Há diferença funcional? Há diferença de sincronização?


# 2. Dependências `in`, `out` e `inout`

Considere o grafo:

```text
        T1
       /  \
      T2  T3
       \  /
        T4
```

T2 e T3 dependem de T1. T4 depende de T2 e T3.


In [ ]:
%%writefile dependencias.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main(void) {
    int x = 0, y = 0, z = 0;

    #pragma omp parallel num_threads(4)
    #pragma omp single
    {
        #pragma omp task depend(out: x)
        {
            usleep(120000);
            x = 10;
            printf("T1: produziu x = %d, thread %d\n", x, omp_get_thread_num());
        }

        #pragma omp task depend(in: x) depend(out: y)
        {
            usleep(80000);
            y = 2 * x;
            printf("T2: consumiu x = %d e produziu y = %d, thread %d\n",
                   x, y, omp_get_thread_num());
        }

        #pragma omp task depend(in: x) depend(out: z)
        {
            usleep(50000);
            z = x + 5;
            printf("T3: consumiu x = %d e produziu z = %d, thread %d\n",
                   x, z, omp_get_thread_num());
        }

        #pragma omp task depend(in: y, z) depend(inout: x)
        {
            x = y + z;
            printf("T4: consumiu y = %d e z = %d; novo x = %d, thread %d\n",
                   y, z, x, omp_get_thread_num());
        }
    }

    printf("Resultado final: x = %d, y = %d, z = %d\n", x, y, z);
    return 0;
}


Overwriting dependencias.c


In [ ]:
!gcc -O2 -fopenmp dependencias.c -o dependencias
!./dependencias


### Questões

1. Por que T2 e T3 podem executar simultaneamente?
2. Por que T4 só pode começar após T2 e T3?
3. A ordem textual das diretivas, isoladamente, garante essa ordem?
4. As threads que executam as tarefas são sempre as mesmas e diferentes entre si?
5. Qual é o resultado final obtido?


## 3 Atividade: completar uma cadeia de dependências

Complete as cláusulas para obter a sequência:

```text
leitura → transformação → validação → gravação
```


In [ ]:
%%writefile pipeline_todo.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    int dados[4] = {0, 0, 0, 0};
    int valido = 0;

    #pragma omp parallel num_threads(6)
    #pragma omp single
    {
        /* TODO: esta tarefa produz dados */
        #pragma omp task
        {
            for (int i = 0; i < 4; i++) dados[i] = i + 1;
            printf("Leitura concluída\n");
        }

        /* TODO: esta tarefa lê e modifica dados */
        #pragma omp task
        {
            for (int i = 0; i < 4; i++) dados[i] *= 10;
            printf("Transformação concluída\n");
        }

        /* TODO: esta tarefa lê dados e faz a validação */
        #pragma omp task
        {
            valido = 1;
            for (int i = 0; i < 4; i++)
                if (dados[i] <= 0) valido = 0;
            printf("Validação concluída: %d\n", valido);
        }

        /* TODO: esta tarefa depende da validação */
        #pragma omp task
        {
            printf("Gravação: ");
            for (int i = 0; i < 4; i++) printf("%d ", dados[i]);
            printf("\n");
        }
    }

    return 0;
}


Overwriting pipeline_todo.c


Depois de completar as cláusulas, compile com:


In [ ]:
!gcc -O2 -fopenmp pipeline_todo.c -o pipeline_todo
!./pipeline_todo

# 4. `barrier` versus `taskwait`

- `barrier`: sincroniza todas as threads da equipe e garante a conclusão das tarefas pendentes antes da saída.
- `taskwait`: apenas a tarefa que encontra a diretiva espera seus filhos diretos.


In [ ]:
%%writefile barrier_taskwait.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel num_threads(6)
    {
        int tid = omp_get_thread_num();

        #pragma omp single nowait
        {
            for (int i = 0; i < 4; i++) {
                #pragma omp task firstprivate(i)
                {
                    usleep(50000 * (4 - i));
                    #pragma omp critical
                    printf("Task %d concluída pela thread %d\n",
                           i, omp_get_thread_num());
                }
            }

            #pragma omp taskwait
            printf("A tarefa single terminou de esperar seus filhos diretos.\n");
        }

        #pragma omp barrier

        #pragma omp critical
        printf("Thread %d passou pela barrier.\n", tid);
    }

    return 0;
}


Overwriting barrier_taskwait.c


In [ ]:
!gcc -O2 -fopenmp barrier_taskwait.c -o barrier_taskwait
!./barrier_taskwait


### Questões

1. Quantas threads encontram `taskwait`?
2. Quantas threads encontram `barrier`?
3. A barreira seria necessária sem `single nowait`?
4. `taskwait` espera tarefas criadas por outras threads?


# 5. `taskwait` versus `taskgroup`

`taskwait` espera apenas filhos diretos. `taskgroup` espera as tarefas de sua região e todos os seus descendentes.


In [ ]:
%%writefile taskwait_taskgroup.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

void com_taskwait(void) {
    printf("\n=== taskwait ===\n");

    #pragma omp parallel num_threads(4)
    #pragma omp single
    {
        #pragma omp task
        {
            printf("Filho direto começou\n");

            #pragma omp task
            {
                usleep(200000);
                printf("Neto terminou\n");
            }

            printf("Filho direto terminou sem esperar o neto\n");
        }

        #pragma omp taskwait
        printf("Taskwait terminou\n");
    }
}

void com_taskgroup(void) {
    printf("\n=== taskgroup ===\n");

    #pragma omp parallel num_threads(4)
    #pragma omp single
    {
        #pragma omp taskgroup
        {
            #pragma omp task
            {
                printf("Filho direto começou\n");

                #pragma omp task
                {
                    usleep(200000);
                    printf("Neto terminou\n");
                }

                printf("Filho direto terminou sem taskwait interno\n");
            }
        }

        printf("Taskgroup terminou\n");
    }
}

int main(void) {
    com_taskwait();
    com_taskgroup();
    return 0;
}


Writing taskwait_taskgroup.c


In [ ]:
!gcc -O2 -fopenmp taskwait_taskgroup.c -o taskwait_taskgroup
!./taskwait_taskgroup


### Questões

1. Na versão com `taskwait`, a mensagem final pode aparecer antes de `Neto terminou`?
2. Por que isso não ocorre com `taskgroup`?
3. `taskwait` espera todos os descendentes?Em que taskgroup difere de `taskwait`?
4. Qual diretiva é mais apropriada quando é necessário aguardar toda uma árvore de tarefas?


# 6. Diretiva `taskyield`

`taskyield` oferece ao runtime um ponto explícito de escalonamento. É uma sugestão: não garante troca de tarefa nem ordem de execução.


In [ ]:
%%writefile taskyield.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel num_threads(2)
    #pragma omp single
    {
        #pragma omp task
        {
            for (int etapa = 1; etapa <= 5; etapa++) {
                #pragma omp critical
                printf("Tarefa longa: etapa %d, thread %d\n",
                       etapa, omp_get_thread_num());

                usleep(30000);
                #pragma omp taskyield
            }
        }

        for (int i = 0; i < 5; i++) {
            #pragma omp task firstprivate(i)
            {
                #pragma omp critical
                printf("Tarefa curta %d, thread %d\n",
                       i, omp_get_thread_num());
            }
        }
    }

    return 0;
}


Writing taskyield.c


In [ ]:
!gcc -O2 -fopenmp taskyield.c -o taskyield
!./taskyield


### Questões

1. O `taskyield` garantiu alternância?
2. Por que o comportamento pode variar entre runtimes?
3. Ele substitui `taskwait`, `taskgroup` ou `depend`?


# 7. Diretiva `taskloop`

`taskloop` divide um laço em blocos e cria tarefas para executar esses blocos.

- `grainsize(g)`: controla aproximadamente o número de iterações por tarefa;
- `num_tasks(t)`: solicita `t` tarefas;
- `nogroup`: remove o `taskgroup` implícito.


In [ ]:
%%writefile taskloop_basico.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    int n = 32;
    int grain = 4;

    if (argc > 1) n = atoi(argv[1]);
    if (argc > 2) grain = atoi(argv[2]);

    int *v = malloc(sizeof(int) * n);
    if (!v) return 1;

    #pragma omp parallel num_threads(4)
    #pragma omp single
    {
        #pragma omp taskloop grainsize(grain) shared(v)
        for (int i = 0; i < n; i++) {
            v[i] = i * i;
            #pragma omp critical
            printf("i = %2d processado pela thread %d\n",
                   i, omp_get_thread_num());
        }
    }

    printf("\nVetor resultante:\n");
    for (int i = 0; i < n; i++) printf("%d ", v[i]);
    printf("\n");

    free(v);
    return 0;
}


Overwriting taskloop_basico.c


In [ ]:
!gcc -O2 -fopenmp taskloop_basico.c -o taskloop_basico
!./taskloop_basico 24 4


### Experimento de granularidade


In [ ]:
!for g in 1 2 4 8 16; do echo "===== grainsize = $g ====="; ./taskloop_basico 32 $g | head -n 12; echo; done


### Questões

1. Um `grainsize` menor tende a criar mais ou menos tarefas?
2. A ordem das iterações é necessariamente crescente?
3. Por que o vetor final continua correto?
4. Qual é o efeito de tarefas excessivamente pequenas?


## 7.1 `num_tasks`


In [ ]:
%%writefile taskloop_num_tasks.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    int n = 24;
    int ntasks = 4;

    if (argc > 1) n = atoi(argv[1]);
    if (argc > 2) ntasks = atoi(argv[2]);

    #pragma omp parallel num_threads(4)
    #pragma omp single
    {
        #pragma omp taskloop num_tasks(ntasks)
        for (int i = 0; i < n; i++) {
            #pragma omp critical
            printf("Iteração %2d, thread %d\n", i, omp_get_thread_num());
        }
    }

    return 0;
}


In [ ]:
!gcc -O2 -fopenmp taskloop_num_tasks.c -o taskloop_num_tasks
!./taskloop_num_tasks 24 4


### Questões

1. `num_tasks(4)` garante quatro threads?
2. Qual é a diferença entre quantidade de tarefas e quantidade de threads?
3. Compare `num_tasks(4)` com `grainsize(6)` para 24 iterações.


## 7.2 `nogroup`


In [ ]:
%%writefile taskloop_nogroup.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <omp.h>

void sem_nogroup(int *v, int n) {
    printf("\n=== taskloop com taskgroup implícito ===\n");

    #pragma omp parallel num_threads(4)
    #pragma omp single
    {
        #pragma omp taskloop grainsize(2)
        for (int i = 0; i < n; i++) {
            usleep(20000);
            v[i] = i + 1;
        }

        printf("Após taskloop: v[0] = %d, v[n-1] = %d\n",
               v[0], v[n - 1]);
    }
}

void com_nogroup(int *v, int n) {
    printf("\n=== taskloop nogroup ===\n");

    #pragma omp parallel num_threads(4)
    #pragma omp single
    {
        #pragma omp taskloop grainsize(2) nogroup
        for (int i = 0; i < n; i++) {
            usleep(20000);
            v[i] = i + 1;
        }

        printf("Imediatamente após nogroup: v[0] = %d, v[n-1] = %d\n",
               v[0], v[n - 1]);

        #pragma omp taskwait

        printf("Após taskwait: v[0] = %d, v[n-1] = %d\n",
               v[0], v[n - 1]);
    }
}

int main(void) {
    int n = 16;
    int *v = calloc(n, sizeof(int));

    sem_nogroup(v, n);
    memset(v, 0, sizeof(int) * n);
    com_nogroup(v, n);

    free(v);
    return 0;
}


Writing taskloop_nogroup.c


In [ ]:
!gcc -O2 -fopenmp taskloop_nogroup.c -o taskloop_nogroup
!./taskloop_nogroup



=== taskloop com taskgroup implícito ===
Após taskloop: v[0] = 1, v[n-1] = 16

=== taskloop nogroup ===
Imediatamente após nogroup: v[0] = 0, v[n-1] = 0
Após taskwait: v[0] = 1, v[n-1] = 16


### Questões

1. Qual sincronização implícita existe no primeiro `taskloop`?
2. O que `nogroup` remove?
3. Por que foi necessário inserir `taskwait`?


# 8. Atividade de programação

Implemente um programa que:

1. crie uma lista encadeada com pelo menos 100 elementos;
2. use uma única thread para percorrer a lista;
3. crie uma tarefa para calcular o quadrado de cada valor;
4. armazene os resultados em um vetor;
5. use `firstprivate` corretamente para preservar nó e índice;
6. sincronize antes de imprimir o vetor;
7. compare execução sequencial e execução com tasks;
8. repita o experimento com trabalhos de granularidades diferentes.

Discuta em quais situações o overhead de criação e escalonamento supera o benefício do paralelismo.
